# 1.4 Prepare evidence and construct a request

Read the matching lesson in `modules/01-standalone-foundations/04-evidence-pipeline.md`. This notebook adds an executable walkthrough; the lesson supplies the full conceptual and organisational explanation.

**How to work:** select the project `.venv` kernel, then run cells from top to bottom. A kernel is the Python process that remembers values between cells. Restart it and run all cells to check that an experiment does not depend on hidden earlier work. Predict results before running a cell. Exercises and worked answers are labelled separately.

All data here are fictional. No API key, provider account or network call is used.


## Load the shared implementation

`import` makes saved functions available in this session. The editable course package locates the checkout, so this cell works even when Jupyter starts in another directory. `sys.path` tells Python where to find the pilot file. The functions themselves are fully explained in the notes and can be opened in `examples/01-foundations/foundations_lab.py`.


In [ ]:
from ai_orgdesign.workspace import project_root
import sys

ROOT = project_root()
LAB_FOLDER = ROOT / "examples" / "01-foundations"
if str(LAB_FOLDER) not in sys.path:
    sys.path.insert(0, str(LAB_FOLDER))
import foundations_lab as lab

print("Course code loaded. This notebook makes no network calls.")


## Preserve disagreement and provenance

A source identifier records where a value came from. It helps an investigator trace a statement, but does not itself prove the statement true. `pprint` formats nested values for inspection.


In [ ]:
from pprint import pprint
import copy
import json

original = copy.deepcopy(lab.CASE)
brief = lab.build_brief(original)
pprint(brief)
assert original == lab.CASE
assert [fact["status"] for fact in brief["facts"]] == ["recorded", "recorded", "reported"]


## Validate before preparing a request

`deepcopy` copies nested records as well as the outer dictionary. Editing a shallow copy could accidentally change the original inner dictionaries. The following deliberately invalid input uses a negative quantity.


In [ ]:
invalid = copy.deepcopy(lab.CASE)
invalid["customer_report"]["quantity"] = -1
try:
    lab.build_brief(invalid)
except ValueError as error:
    print("Expected validation failure:", error)
else:
    raise AssertionError("Negative quantities must be rejected.")


## Serialise messages

Serialisation converts structured values into text for storage or transmission. `json.loads` reverses the JSON text representation here. These cells only construct messages; no SDK, provider or neural model is called. Instructions inside a message are not a security guarantee.


In [ ]:
messages = lab.make_request(brief)
print(messages[0]["content"])
recovered = json.loads(messages[1]["content"])
assert recovered == brief
print(recovered["unresolved"])


## Exercise

Change the customer report to twelve. Does agreement remove the need to establish what actually happened?

## Worked answer

The difference becomes zero, but the records still describe recorded or reported quantities. Agreement is useful evidence, not independent verification. This lab conservatively retains its investigation status and unresolved fields even when quantities agree. A production service would need explicit, justified closure rules.


In [ ]:
agreement_case = copy.deepcopy(lab.CASE)
agreement_case["customer_report"]["quantity"] = 12
agreement = lab.build_brief(agreement_case)
assert agreement["comparison"]["order_minus_customer_report"] == 0
assert "actual_received_quantity" in agreement["unresolved"]
pprint(agreement["comparison"])
